## **SNAP Jupyter demo notebook**
**SAR + optical collocation — fuse Sentinel-1 backscatter with Sentinel-2 spectral indices**

In summary, this workflow contains:

- Background on why combining SAR and optical helps
- Preprocess a Sentinel-1 GRD scene to terrain-corrected sigma₀ (VV, VH)
- Preprocess a Sentinel-2 L2A scene: resample to 10 m, compute NDVI and NDWI
- **Collocate** the S1 product onto the S2 grid
- Compute a joint feature (RGB composite of NDVI + sigma₀_VH + sigma₀_VV) and a simple wetland/crop application

Complexity: advanced

##### ***About the test data:***

One Sentinel-1 GRD product and one Sentinel-2 L2A product covering the **same area**, on **close dates** (ideally within a few days). Both downloadable from [Copernicus Browser](https://dataspace.copernicus.eu/browser/).

- Sentinel-1: `S1*_IW_GRDH_1S*` (IW mode is most common; SM also works)
- Sentinel-2: `S2*_MSIL2A_*` (L2A = atmospherically corrected; L1C also works with worse NDVI/NDWI quality)

Place both products under `data/` and update the path variables below.

Pick a scene with a mix of vegetation, water and bare ground for an interesting fused result.

##### ***Some information on the Python environment:***

In [ ]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

##### ***Import Python packages...***

In [ ]:
import esa_snappy
from esa_snappy import ProductIO

import snapista
from snapista import Graph
from snapista import Operator

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Convenience plot functions:***

In [ ]:
def _read_band(product, name):
    band = product.getBand(name)
    if band is None:
        raise KeyError(f"Band {name!r} not found. Available: {[b.getName() for b in product.getBands()]}")
    w, h = band.getRasterWidth(), band.getRasterHeight()
    data = np.zeros(w * h, np.float32)
    band.readPixels(0, 0, w, h, data)
    data.shape = h, w
    return data

def plot_band(product, name, title=None, cmap='viridis', vmin=None, vmax=None):
    data = _read_band(product, name)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title or name)
    fig.colorbar(im, ax=ax)
    plt.show()

def plot_interferogram(product, phase_band_name, coh_band_name=None):
    phi = _read_band(product, phase_band_name)
    if coh_band_name is None:
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title('Wrapped interferogram phase [rad]')
        fig.colorbar(im, ax=ax)
    else:
        coh = _read_band(product, coh_band_name)
        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        im0 = axs[0].imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        axs[0].set_title('Wrapped phase [rad]')
        fig.colorbar(im0, ax=axs[0])
        im1 = axs[1].imshow(coh, cmap='viridis', vmin=0, vmax=1)
        axs[1].set_title('Coherence')
        fig.colorbar(im1, ax=axs[1])
    plt.show()

def find_band(product, *patterns):
    names = [b.getName() for b in product.getBands()]
    for pat in patterns:
        for n in names:
            if pat.lower() in n.lower():
                return n
    raise KeyError(f"No band matching {patterns!r} found. Available: {names}")

---

### ***Background: why combine SAR and optical?***

The two sensor types see different physical properties of the surface and have complementary failure modes:

| Property | Sentinel-1 (SAR) | Sentinel-2 (optical) |
|:---------|:-----------------|:---------------------|
| Day/night | yes | day only |
| Cloud cover | sees through | blocked |
| Sensitive to | dielectric, roughness, structure | chlorophyll, water, soil colour |
| Typical issues | speckle, geometric distortion, layover | clouds, shadows, atmospheric haze |
| Volume scattering | strong over canopy | weak |
| Vegetation greenness | weak | strong (NDVI) |
| Surface roughness | strong | weak |

Joint analysis works because the *combined* feature space discriminates classes that neither sensor can separate alone. Two canonical examples:

- **Flooded vegetation**: water under canopy is invisible to S2 (canopy blocks the view) but shows up as a strong double-bounce return in S1.
- **Crop type classification**: NDVI tracks chlorophyll; sigma₀_VH tracks biomass. Two crops with similar NDVI but different structure (e.g. wheat vs rapeseed) separate cleanly on the S1 axis.

The SNAP operator `Collocate` resamples one product onto another's geocoded grid, producing a single multi-band product where every pixel has both sensors' values at the same lat/lon.

---

### ***Configure input paths***

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

s1_grd = os.path.join(data_dir, 'S1A_IW_GRDH_1SDV_<acquisition>.SAFE', 'manifest.safe')
s2_l2a = os.path.join(data_dir, 'S2A_MSIL2A_<acquisition>.SAFE', 'MTD_MSIL2A.xml')

---
## ***Part 1 — Preprocess the Sentinel-1 GRD***
---

Standard GRD prep: orbit, thermal noise + border noise removal, calibration, a Refined Lee speckle filter, and Range-Doppler terrain correction to put the data in a map projection on a 10 m grid (so it'll match the S2 grid after resampling). Output bands: `Sigma0_VV`, `Sigma0_VH`.

In [ ]:
g_s1 = Graph()
g_s1.add_node(operator=Operator('Read', file=s1_grd), node_id='Read')
g_s1.add_node(operator=Operator('Apply-Orbit-File',
                                orbitType='Sentinel Precise (Auto Download)',
                                continueOnFail='true'),
              node_id='Orbit', source='Read')
g_s1.add_node(operator=Operator('ThermalNoiseRemoval',
                                selectedPolarisations='VV,VH'),
              node_id='ThermalNoise', source='Orbit')
g_s1.add_node(operator=Operator('Remove-GRD-Border-Noise',
                                selectedPolarisations='VV,VH'),
              node_id='BorderNoise', source='ThermalNoise')
g_s1.add_node(operator=Operator('Calibration',
                                outputSigmaBand='true',
                                selectedPolarisations='VV,VH'),
              node_id='Calib', source='BorderNoise')
g_s1.add_node(operator=Operator('Speckle-Filter',
                                filter='Refined Lee',
                                windowSize='5x5'),
              node_id='Speckle', source='Calib')
g_s1.add_node(operator=Operator('Terrain-Correction',
                                demName='Copernicus 30m Global DEM',
                                mapProjection='WGS84(DD)',
                                pixelSpacingInMeter='10',
                                imgResamplingMethod='BILINEAR_INTERPOLATION'),
              node_id='TC', source='Speckle')

s1_out = os.path.join(results_dir, 'snap_nb_sar_optical_s1.dim')
g_s1.add_node(operator=Operator('Write', file=s1_out, formatName='BEAM-DIMAP'),
              node_id='Write', source='TC')

g_s1.save_graph(os.path.join(graphs_dir, 'snap_nb_sar_optical_s1.xml'))
g_s1.run()

---
## ***Part 2 — Preprocess the Sentinel-2 L2A***
---

S2 L2A has bands at 10 m (B2 B3 B4 B8), 20 m (B5 B6 B7 B8A B11 B12) and 60 m (B1 B9). `Resample` puts everything on the same grid (10 m here). Then `BandMaths` computes NDVI and NDWI.

In [ ]:
g_s2 = Graph()
g_s2.add_node(operator=Operator('Read', file=s2_l2a), node_id='Read')
g_s2.add_node(operator=Operator('Resample',
                                referenceBand='B4',
                                upsampling='Bilinear',
                                downsampling='Mean'),
              node_id='Resample', source='Read')

# NDVI = (NIR - Red) / (NIR + Red)
# NDWI = (Green - NIR) / (Green + NIR)   (McFeeters 1996)
g_s2.add_node(operator=Operator('BandMaths',
                                targetBandDescriptors=[
                                    {'name': 'NDVI',
                                     'type': 'float32',
                                     'expression': '(B8 - B4) / (B8 + B4)'},
                                    {'name': 'NDWI',
                                     'type': 'float32',
                                     'expression': '(B3 - B8) / (B3 + B8)'},
                                ]),
              node_id='Indices', source='Resample')
# Note: this dict-based BandMaths syntax may need adjustment for your SNAPISTA version.
# If it errors, see the OLCI band-maths notebook for the TargetBandDescriptors pattern.

s2_out = os.path.join(results_dir, 'snap_nb_sar_optical_s2.dim')
g_s2.add_node(operator=Operator('Write', file=s2_out, formatName='BEAM-DIMAP'),
              node_id='Write', source='Indices')

g_s2.save_graph(os.path.join(graphs_dir, 'snap_nb_sar_optical_s2.xml'))
g_s2.run()

---
## ***Part 3 — Collocate***
---

`Collocate` takes two source products: a **master** (the target grid) and a **slave** (which gets resampled). The output has both products' bands on the master grid, with slave bands renamed to avoid collisions.

We pick **S2 as master** because we want the result in the S2 10 m UTM grid — this matches common downstream tools and S2's CRS is well-defined per scene.

In [ ]:
g_coloc = Graph()
g_coloc.add_node(operator=Operator('Read', file=s2_out), node_id='ReadS2')
g_coloc.add_node(operator=Operator('Read', file=s1_out), node_id='ReadS1')

g_coloc.add_node(operator=Operator('Collocate',
                                   renameMasterComponents='false',
                                   renameSlaveComponents='true',
                                   slaveComponentPattern='${ORIGINAL_NAME}_S1',
                                   resamplingType='BILINEAR_INTERPOLATION'),
                node_id='Collocate',
                source=['ReadS2', 'ReadS1'])
# Note: Snapista may pass master/slave in `source` order. If your version expects
# masterProductName / slaveProductName as parameters, switch to that form.

coloc_out = os.path.join(results_dir, 'snap_nb_sar_optical_collocated.dim')
g_coloc.add_node(operator=Operator('Write', file=coloc_out, formatName='BEAM-DIMAP'),
                 node_id='Write', source='Collocate')

g_coloc.save_graph(os.path.join(graphs_dir, 'snap_nb_sar_optical_collocated.xml'))
g_coloc.run()

In [ ]:
p_coloc = ProductIO.readProduct(coloc_out)
print('Bands in collocated product:')
for b in p_coloc.getBands():
    print(f'  {b.getName():30s} unit={b.getUnit()}')
print('Width x Height:', p_coloc.getSceneRasterWidth(), 'x', p_coloc.getSceneRasterHeight())

---
## ***Part 4 — Joint feature: SAR-optical RGB composite***
---

An informative quick-look that puts SAR and optical on the same colour axes: compose an RGB image with

- **Red** = sigma₀_VH (dB stretched) — SAR cross-pol, sensitive to volume scattering
- **Green** = NDVI — vegetation greenness
- **Blue** = NDWI — surface water

Anything bright red but low green = scattering structure without much chlorophyll (buildings, dry shrubs). Bright green but low red = lush vegetation with little physical structure (grass, crops). Bright blue = open water. Multi-channel bright = complex surface.

In [ ]:
vh_name = find_band(p_coloc, 'Sigma0_VH_S1', 'Sigma0_VH')
ndvi_name = find_band(p_coloc, 'NDVI')
ndwi_name = find_band(p_coloc, 'NDWI')

vh = _read_band(p_coloc, vh_name)
vh_db = 10.0 * np.log10(np.maximum(vh, 1e-6))
ndvi = _read_band(p_coloc, ndvi_name)
ndwi = _read_band(p_coloc, ndwi_name)

# Independent stretches per channel
def stretch(arr, lo=2, hi=98):
    a, b = np.nanpercentile(arr, [lo, hi])
    return np.clip((arr - a) / (b - a + 1e-9), 0, 1)

r = stretch(vh_db)
g = stretch(ndvi)
b = stretch(ndwi)
rgb = np.stack([r, g, b], axis=-1)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(rgb)
ax.set_title('SAR-optical composite — R: σ₀_VH dB, G: NDVI, B: NDWI')
plt.show()

##### ***A simple thematic application: "wet vegetation" mask:***

Wet vegetation = high NDVI **and** moderately high backscatter — the SAR sees structure, the optical sees chlorophyll, and the two together can flag features like irrigated crops, mangroves, or flooded forest that neither sensor on its own would identify.

In [ ]:
# Per-pixel rules — thresholds chosen for illustration; tune to your scene
wet_veg = (ndvi > 0.4) & (vh_db > -18) & (vh_db < -10)

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].imshow(rgb)
axs[0].set_title('SAR-optical composite')
axs[1].imshow(wet_veg, cmap='Greens')
axs[1].set_title('Wet vegetation mask (NDVI > 0.4 & -18 < VH_dB < -10)')
plt.show()

p_coloc.dispose()

---

### ***Summary***

What have we learnt in this notebook?

- The standard SAR-optical fusion workflow is: preprocess each sensor independently into a terrain-corrected map-projected product, then `Collocate` them onto a shared grid.
- Picking **S2 as the master** keeps the result in S2's well-defined per-scene UTM grid and 10 m spacing — convenient for downstream tools.
- `Collocate`'s `slaveComponentPattern` lets you rename incoming bands to avoid collisions and to make later `BandMaths` expressions unambiguous.
- A simple **R = σ₀_VH dB / G = NDVI / B = NDWI** RGB composite is a surprisingly informative quick-look for any agriculture, wetland or urban-edge scene.
- Real applications combine these channels through thresholds, classifiers or decision rules — we showed the simplest case: a per-pixel rule for "wet vegetation".